In [2]:
import coiled
import fsspec
import s3fs
import numpy as np
import rioxarray
import xarray as xr
import fsspec
import pandas as pd
import logging 
import numpy as np
import pytz
import dask
import re
import requests
import sparse
import time
import warnings
import zarr
from io import BytesIO
from datetime import datetime
from dask.distributed import Client, LocalCluster
from dask.distributed import print
from flox import ReindexArrayType, ReindexStrategy
from flox.xarray import xarray_reduce
import pygwalker as pyg

# T0 INSTALL FLOX:
# Downloaded flox 0.10.3 from https://pypi.org/project/flox/#files usin the source distribution (tar.gz, https://files.pythonhosted.org/packages/0b/b6/5e3d79ef8e3dd3bb1ba656167e2059c0aa000244321995c508db32c7a578/flox-0.10.3.tar.gz)
# Installed in my working conda environment using pip: pip install /home/dagibbs22/flox-0.10.3.tar.gz

# TO CREATE A NOTEBOOK IN A COILED CLUSTER
# coiled notebook start --region=us-east-1

In [ ]:
# Zarr creation cluster
cluster = coiled.Cluster(
    name="LULUCF_zarr_creation",
    region="us-east-1", # close to dataset, avoid egress charges
    n_workers=5,
    tags={"project": "LULUCF_zonal_stats"},
    scheduler_vm_types="r7g.xlarge", 
    worker_vm_types="r7g.2xlarge",
    compute_purchase_option="spot_with_fallback"
)

client = cluster.get_client()

In [129]:
# Zonal stats cluster
cluster = coiled.Cluster(
    name="LULUCF_zonal_stats",
    region="us-east-1", # close to dataset, avoid egress charges
    n_workers=20,
    tags={"project": "LULUCF_zonal_stats"},
    scheduler_vm_types="r7g.xlarge", 
    worker_vm_types="r7g.2xlarge",
    compute_purchase_option="spot_with_fallback"
)

client = cluster.get_client()

Output()

╭──────────────────────────────── Package Info ────────────────────────────────╮
│                ╷                                                             │
│   Package      │ Note                                                        │
│ ╶──────────────┼───────────────────────────────────────────────────────────╴ │
│   flox         │ Wheel built from ~/flox-0.10.3.tar.gz                       │
│                ╵                                                             │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Not Synced with Cluster ───────────────────────────╮
│                 ╷                                                ╷           │
│   Package       │ Error                                          │ Level     │
│ ╶───────────────┼────────────────────────────────────────────────┼─────────╴ │
│   pygwalker     │ Pip check had the following issues that need   │ Warning   │
│                 │ resolving:                                     │           │
│                 │ pygwalker 0.3.17 has requirement               │           │
│                 │ duckdb==0.9.2, but you have duckdb 1.3.0.      │           │
│                 │ pygwalker 0.3.17 has requirement               │           │
│                 │ segment-analytics-python==2.2.3, but you have  │           │
│                 │ segment-analytics-python 2.3.3.                │           │
│   pydantic_core │ pydantic-core~=2.33.2 has no install candidate │ Warning   │
│                 │ for Python 3.12 linux-aarch64 on conda-forge   │           │
│   dtale         │ Pip check had the following issues that need   │ Warning   │
│                 │ resolving:                                     │           │
│                 │ dtale 3.17.0 has requirement dash<=2.18.2;     │           │
│                 │ python_version > "3.7", but you have dash      │           │
│                 │ 3.0.4.                                         │           │
│                 │ dtale 3.17.0 has requirement                   │           │
│                 │ dash-bootstrap-components<=1.7.1;              │           │
│                 │ python_version > "3.0", but you have           │           │
│                 │ dash-bootstrap-components 2.0.3.               │           │
│                 │ dtale 3.17.0 has requirement dash_daq<=0.5.0,  │           │
│                 │ but you have dash-daq 0.6.0.                   │           │
│   awscrt        │ awscrt~=0.26.1 has no install candidate for    │ Warning   │
│                 │ Python 3.12 linux-aarch64 on conda-forge       │           │
│                 ╵                                                ╵           │
╰──────────────────────────────────────────────────────────────────────────────╯

Output()

In [ ]:
client.restart() 

In [ ]:
local_cluster = LocalCluster()  
local_client = Client(local_cluster)
local_client

In [ ]:
local_client.shutdown()

In [4]:
# To hide the warning "UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future."
warnings.filterwarnings(
    "ignore",
    message="Consolidated metadata is currently not part in the Zarr format 3 specification.*",
    category=UserWarning,
    module="zarr.api.asynchronous"
)

# Conversion of carbon to CO2
C_to_CO2 = 44/12

def timestr():

    # Define the Eastern Time timezone
    eastern = pytz.timezone('US/Eastern')

    # Get the current time in UTC and convert to Eastern Time
    eastern_time = datetime.now(eastern)

    # Format the time as a string
    return eastern_time.strftime("%Y%m%d_%H_%M_%S")

In [5]:
# Creates a Pandas dataframe with the state_nodes codes and meanings from an Excel spreadsheet
def create_state_node_df(state_node_lookup_table_local, state_node_lookup_table_s3, sheet_name):

    try:
        # Tries fetching the file from the S3 URL
        # print(f"Attempting to download file from URL: {spreadsheet}")
        response = requests.get(state_node_lookup_table_s3, timeout=10)
        response.raise_for_status()
        state_node_df = pd.read_excel(BytesIO(response.content), sheet_name=sheet_name)

    except (requests.exceptions.RequestException, Exception) as e:
        print(f"Failed to download file from S3. Falling back to local file. Error: {e}")

        print(f"Reading file from local path: {state_node_lookup_table_local}")
        state_node_df = pd.read_excel(state_node_lookup_table_local, sheet_name=sheet_name)

    return state_node_df

In [6]:
# Lists uris in an s3 folder, for creating zarr of them
# per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/682201ec-1f84-800a-a9f9-c9564f613208
def list_folder_uris(base_uri):

    # Initializes S3 filesystem
    fs = s3fs.S3FileSystem(anon=False)  # Set anon=True if public bucket
    
    # Lists all files in the directory
    all_files = fs.ls(base_uri)
    
    # Filters for GeoTIFFs
    tif_files = [f"s3://{f}" for f in all_files if f.endswith(".tif")]
    
    # Converts to a Pandas Series
    series = pd.Series(tif_files)
    
    return series

# Extracts file pattern from uri. Assumes that file pattern includes _ha_yr (as it does from the LULUCF model).
def parse_pattern_from_uri(uri_series):

    uri = uri_series.values.tolist()[0]
    # print("Parsing URI:", uri)

    # regex per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/681a538d-55e4-800a-818b-bcf850757ba0
    pattern = r"__([a-zA-Z0-9_]+(?:__?[a-zA-Z0-9_]+)*)_ha_yr_\d{4}_\d{4}\.tif$"
    match = re.search(pattern, uri)

    if match:
        return match.group(1)
    else:
        return None

In [7]:
# Makes xarray dataframe (I think not a dataset) from list of s3 uris.
# This came from Solomon Negusse and I haven't really changed it.
# He said that an online forum suggested using xr.openmfdataset to open non-overlapping geotifs.
def make_xarray_chunks(tile_uris, chunk_size):

    xarray_chunks = xr.open_mfdataset(
        tile_uris.values.tolist(),
        parallel=True,
        chunks={'x': chunk_size, 'y':chunk_size}
    ).squeeze()
    # ).squeeze().persists()  # Need this if reading from geotifs directly, rather then creating zarrs

    return xarray_chunks

In [8]:
# Function to check for grid consistency of model outputs within a single interval
# From https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6899f498-9e9c-8325-aecb-d1d8df7fe48f
def assert_same_grid(a, b, name_a, name_b):
    if a.sizes['x'] != b.sizes['x'] or a.sizes['y'] != b.sizes['y']:
        raise ValueError(f"Shape mismatch between {name_a} and {name_b}")
    if not np.array_equal(a['x'].values, b['x'].values) or not np.array_equal(a['y'].values, b['y'].values):
        raise ValueError(f"Coordinate mismatch between {name_a} and {name_b}")

In [9]:
# Writes a new zarr or appends to existing zarr.
# From https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6899f498-9e9c-8325-aecb-d1d8df7fe48f
# Retries appending if there's a problem for safety. 
def write_or_append_with_retry(ds: xr.Dataset, zarr_path: str, *, append: bool, max_retries=3, wait_seconds=5):
    """Write a dataset to a Zarr (create or append along 'interval') with retries."""
    attempt = 1
    while True:
        if attempt > 1:
            print(f"On second attempt to make zarr for {zarr_path}")
        try:
            if append:
                ds.to_zarr(
                    zarr_path,
                    mode='a',
                    zarr_format=3,                  
                    storage_options={"anon": False},
                    append_dim='interval',
                )
            else:
                ds.to_zarr(
                    zarr_path,
                    mode='w',
                    zarr_format=3,                  
                    storage_options={"anon": False},
                )
            return
        except Exception as e:
            if attempt >= max_retries:
                print(f"   Failed {'append' if append else 'create'} after {attempt} attempts")
                raise
            print(f"   {'Append' if append else 'Create'} failed (attempt {attempt}): {e}")
            attempt += 1
            time.sleep(wait_seconds)

In [10]:
# Crops one input to the other input's extent.
# ref is the reference dataset that is being cropped to. 
# From long chat in https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/684749fe-7b30-800a-ba8b-c502377f2c3a
def safe_crop(ds, ref):
    return ds.sel(x=ref.x, y=ref.y, method="nearest")

In [11]:
### Options for contextual layer values.
### Every contextual layer needs to have all possible values listed here.

# GADM v4.1 adm0 IDs (from Solomon Negusse's notebook)
gadm_adm0_ids = np.array([  0.,   4.,   8.,  10.,  12.,  16.,  20.,  24.,  28.,  31.,  32.,
        36.,  40.,  44.,  48.,  50.,  51.,  52.,  56.,  60.,  64.,  68.,
        70.,  72.,  74.,  76.,  84.,  86.,  90.,  92.,  96., 100., 104.,
       108., 112., 116., 120., 124., 132., 136., 140., 144., 148., 152.,
       156., 158., 162., 166., 170., 174., 175., 178., 180., 184., 188.,
       191., 192., 196., 203., 204., 208., 212., 214., 218., 222., 226.,
       231., 232., 233., 234., 238., 239., 242., 246., 248., 250., 254.,
       258., 260., 262., 266., 268., 270., 275., 276., 288., 292., 296.,
       300., 304., 308., 312., 316., 320., 324., 328., 332., 334., 336.,
       340., 348., 352., 356., 360., 364., 368., 372., 376., 380., 384.,
       388., 392., 398., 400., 404., 408., 410., 414., 417., 418., 422.,
       426., 428., 430., 434., 438., 440., 442., 450., 454., 458., 462.,
       466., 470., 474., 478., 480., 484., 492., 496., 498., 499., 500.,
       504., 508., 512., 516., 520., 524., 528., 531., 533., 534., 535.,
       540., 548., 554., 558., 562., 566.,70., 574., 578., 580., 581.,
       583., 584., 585., 586., 591., 598., 600., 604., 608., 612., 616.,
       620., 624., 626., 630., 634., 638., 642., 643., 646., 652., 654.,
       659., 660., 662., 663., 666., 670., 674., 678., 682., 686., 688.,
       690., 694., 702., 703., 704., 705., 706., 710., 716., 724., 728.,
       729., 732., 740., 744., 748., 752., 756., 760., 762., 764., 768.,
       772., 776., 780., 784., 788., 792., 795., 796., 798., 800., 804.,
       807., 818., 826., 831., 832., 833., 834., 840., 850., 854., 858.,
       860., 862., 876., 882., 887., 894.], dtype=np.uint16)

# Primary forest value options
primary_forest_IFL_codes = np.array([0, 1], dtype=np.uint8)

# Converts numeric ISO values to ISO codes
# From https://github.com/wri/project-zeno-data-infra/blob/main/notebooks/grasslands_areas_gadm_2000-2022.ipynb
numeric_to_alpha3 = {
    4: 'AFG', 248: 'ALA', 8: 'ALB', 12: 'DZA', 16: 'ASM', 20: 'AND', 24: 'AGO', 660: 'AIA',
    10: 'ATA', 28: 'ATG', 32: 'ARG', 51: 'ARM', 533: 'ABW', 36: 'AUS', 40: 'AUT', 31: 'AZE',
    44: 'BHS', 48: 'BHR', 50: 'BGD', 52: 'BRB', 112: 'BLR', 56: 'BEL', 84: 'BLZ', 204: 'BEN',
    60: 'BMU', 64: 'BTN', 68: 'BOL', 535: 'BES', 70: 'BIH', 72: 'BWA', 74: 'BVT', 76: 'BRA',
    86: 'IOT', 96: 'BRN', 100: 'BGR', 854: 'BFA', 108: 'BDI', 132: 'CPV', 116: 'KHM', 120: 'CMR',
    124: 'CAN', 136: 'CYM', 140: 'CAF', 148: 'TCD', 152: 'CHL', 156: 'CHN', 162: 'CXR', 166: 'CCK',
    170: 'COL', 174: 'COM', 178: 'COG', 180: 'COD', 184: 'COK', 188: 'CRI', 384: 'CIV', 191: 'HRV',
    192: 'CUB', 531: 'CUW', 196: 'CYP', 203: 'CZE', 208: 'DNK', 262: 'DJI', 212: 'DMA', 214: 'DOM',
    218: 'ECU', 818: 'EGY', 222: 'SLV', 226: 'GNQ', 232: 'ERI', 233: 'EST', 748: 'SWZ', 231: 'ETH',
    238: 'FLK', 234: 'FRO', 242: 'FJI', 246: 'FIN', 250: 'FRA', 254: 'GUF', 258: 'PYF', 260: 'ATF',
    266: 'GAB', 270: 'GMB', 268: 'GEO', 276: 'DEU', 288: 'GHA', 292: 'GIB', 300: 'GRC', 304: 'GRL',
    308: 'GRD', 312: 'GLP', 316: 'GUM', 320: 'GTM', 831: 'GGY', 324: 'GIN', 624: 'GNB', 328: 'GUY',
    332: 'HTI', 334: 'HMD', 336: 'VAT', 340: 'HND', 344: 'HKG', 348: 'HUN', 352: 'ISL', 356: 'IND',
    360: 'IDN', 364: 'IRN', 368: 'IRQ', 372: 'IRL', 833: 'IMN', 376: 'ISR', 380: 'ITA', 388: 'JAM',
    392: 'JPN', 832: 'JEY', 400: 'JOR', 398: 'KAZ', 404: 'KEN', 296: 'KIR', 408: 'PRK', 410: 'KOR',
    414: 'KWT', 417: 'KGZ', 418: 'LAO', 428: 'LVA', 422: 'LBN', 426: 'LSO', 430: 'LBR', 434: 'LBY',
    438: 'LIE', 440: 'LTU', 442: 'LUX', 446: 'MAC', 450: 'MDG', 454: 'MWI', 458: 'MYS', 462: 'MDV',
    466: 'MLI', 470: 'MLT', 584: 'MHL', 474: 'MTQ', 478: 'MRT', 480: 'MUS', 175: 'MYT', 484: 'MEX',
    583: 'FSM', 498: 'MDA', 492: 'MCO', 496: 'MNG', 499: 'MNE', 500: 'MSR', 504: 'MAR', 508: 'MOZ',
    104: 'MMR', 516: 'NAM', 520: 'NRU', 524: 'NPL', 528: 'NLD', 540: 'NCL', 554: 'NZL', 558: 'NIC',
    562: 'NER', 566: 'NGA', 570: 'NIU', 574: 'NFK', 807: 'MKD', 580: 'MNP', 578: 'NOR', 512: 'OMN',
    586: 'PAK', 585: 'PLW', 275: 'PSE', 591: 'PAN', 598: 'PNG', 600: 'PRY', 604: 'PER', 608: 'PHL',
    612: 'PCN', 616: 'POL', 620: 'PRT', 630: 'PRI', 634: 'QAT', 638: 'REU', 642: 'ROU', 643: 'RUS',
    646: 'RWA', 652: 'BLM', 654: 'SHN', 659: 'KNA', 662: 'LCA', 663: 'MAF', 666: 'SPM', 670: 'VCT',
    882: 'WSM', 674: 'SMR', 678: 'STP', 682: 'SAU', 686: 'SEN', 688: 'SRB', 690: 'SYC', 694: 'SLE',
    702: 'SGP', 534: 'SXM', 703: 'SVK', 705: 'SVN', 90: 'SLB', 706: 'SOM', 710: 'ZAF', 239: 'SGS',
    728: 'SSD', 724: 'ESP', 144: 'LKA', 729: 'SDN', 740: 'SUR', 744: 'SJM', 752: 'SWE', 756: 'CHE',
    760: 'SYR', 158: 'TWN', 762: 'TJK', 834: 'TZA', 764: 'THA', 626: 'TLS', 768: 'TGO', 772: 'TKL',
    776: 'TON', 780: 'TTO', 788: 'TUN', 792: 'TUR', 795: 'TKM', 796: 'TCA', 798: 'TUV', 800: 'UGA',
    804: 'UKR', 784: 'ARE', 826: 'GBR', 840: 'USA', 581: 'UMI', 858: 'URY', 860: 'UZB', 548: 'VUT',
    862: 'VEN', 704: 'VNM', 92: 'VGB', 850: 'VIR', 876: 'WLF', 732: 'ESH', 887: 'YEM', 894: 'ZMB',
    716: 'ZWE'
}

In [12]:
# Converts results of flox to coordinate dictionary.
# This code came from Solomon Negusse and I haven't changed it in any substantial way.
def convert_to_coord_dict(flux_results, interval):

    print(f"   Postprocessing {interval}: {timestr()}")
    sparse_data = flux_results.data
    
    dim_names = flux_results.dims
    indices = sparse_data.coords  # tuple of arrays with indices into each dim
    values = sparse_data.data     # non-zero values
    
    coord_dict = {
        dim: flux_results.coords[dim].values[indices[i]]
        for i, dim in enumerate(dim_names)
    }
    coord_dict["value"] = values

    return coord_dict

In [77]:
# Converts flox output to dataframe and does some processing of it:
# replaces the numeric flux type with the name
# classifies specific flux types to larger groupings
# adds the interval end year to the dataframe
# adds the state node meaning to the dataframe
# converts area from m^2 to ha
def create_interval_df(coord_dict, state_node_df, analysis_layer_dict, interval_length, interval_end_year):

    df = pd.DataFrame(coord_dict)
    # print(df)

    # Replaces numeric values for output flux types with their names for ease of interpretation
    df['analysis_layer'] = df['analysis_layer'].replace(analysis_layer_dict)
    # print("with analysis_layer:", df)

    # Adds the interval end year to the dataframe
    df['interval_end'] = interval_end_year
    # print("with interval end year:", df)

    # Adds the state_node meaning and classifications to the dataframe
    df = df.merge(state_node_df[['state_nodes', 'meaning', 'broad_class', 'detailed_class']],
              left_on='state_nodes', right_on='state_nodes',
              how='left')
    # print("merged:", df)

    # # Converts the area (ha)/interval to ha/yr
    # condition = df['analysis_layer'] == area_year
    # df.loc[condition, 'value'] = df.loc[condition, 'value'] / interval_length

    # Converts numeric codes to ISO codes 
    # From https://github.com/wri/project-zeno-data-infra/blob/main/notebooks/grasslands_areas_gadm_2000-2022.ipynb
    df['gadm_adm0'] = df.gadm_adm0.apply(lambda x: numeric_to_alpha3[x])

    out_df = df

    # Expands table to yearly rows for 5-year intervals: interval_end_year, interval_end_year-1, ..., for interval_length years
    if interval_length == 5:
        years = [interval_end_year - i for i in range(1, 5)]

        # Iterates through years and creates a new dataframe for each output year during the interval with the interval_end_year
        for year in years:
            dfs = df.assign(interval_end=year)
            out_df = pd.concat([dfs, out_df])
    
        # Resets index
        out_df = out_df.reset_index(drop=True)
    
        return out_df

    # Returns annual intervals as-is    
    else:
        return out_df

In [108]:
# Calculates flux densities (Mg CO2 or CO2e/ha) for each output flux
# Per https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/682a8c76-0618-800a-a201-18fd404a281f
def calculate_interval_flux_densities(df):

    # Step 1: Filters out area and flux data
    area_df = df[df['analysis_layer'] == pixel_area_interval].copy()
    flux_df = df[df['analysis_layer'] != pixel_area_interval].copy()
    # print(flux_df.shape[0])
    # print(area_df.shape[0])
    
    # Step 2: Merges flux data with area data on matching keys (contextual_layer_names)
    merged = pd.merge(
        flux_df,
        area_df[contextual_layer_names + ['interval_end', 'value']],
        on=contextual_layer_names + ['interval_end'],
        how='left',
        suffixes=('', '_area')
    )
    # print("merged before area/yr:", merged)
   
    # Step 3: Computes per-hectare flux (converts CO2 to C)
    merged['value_per_ha'] = merged['value'] / merged['value_area'] / C_to_CO2
    
    # Step 4: Prepares flux density rows to append
    new_rows = merged.copy()
    new_rows['analysis_layer'] = new_rows['analysis_layer'] + '__C_per_ha'
    new_rows['value'] = new_rows['value_per_ha']
    new_rows = new_rows.drop(columns=['value_area', 'value_per_ha'])
    # print("new rows:", new_rows)
    
    # Step 5: Appends flux density rows to original dataframe
    result_df = pd.concat([df, new_rows], ignore_index=True)

    return result_df

Code to run zonal stats

In [15]:
# General zonal stat run properties

model_version = "version_0_4_2"  # model version, from s3 paths that are being read
run_date = "20250806"   # model run date, from s3 paths that are being read
chunk_size = 10000  # pixels

# s3 folders for model outputs being analyzed
output_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/{model_version}/"  # Model output path, for inputs to zonal stats

# Analysis layer s3 paths
gross_emis_CO2_folder = f"{output_path}gross_emissions__all_C_pools__CO2_only__MgCO2/standard_model/hybrid_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
gross_emis_all_gases_folder = f"{output_path}gross_emissions__all_C_pools__all_gases__MgCO2e/standard_model/hybrid_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
gross_remv_all_pools_folder = f"{output_path}gross_removals__all_C_pools__MgCO2/standard_model/hybrid_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
net_flux_all_pools_CO2_folder = f"{output_path}net_flux__all_C_pools__CO2_only__MgCO2/standard_model/hybrid_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
net_flux_all_pools_all_gases_folder = f"{output_path}net_flux__all_C_pools__all_gases__MgCO2e/standard_model/hybrid_intervals/INTERVAL/_ha_yr/4000_pixels/{run_date}/"
node_folder = f"{output_path}land_state_node/standard_model/hybrid_intervals/INTERVAL/4000_pixels/{run_date}/"

# Folder where the model output zarrs are stored. They are in their own special outputs folder (at least for now-- we could change this)
zarr_s3_path = f"s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/{model_version}_1x1_inputs_per_ha_all_yrs_combined_zarr/zarr/{run_date}/"
all_intervals_zarr = f"{zarr_s3_path}all_intervals/afolu_flux__all_layers__all_intervals.zarr"

# zarrs for layers not from the flux model (only need to created once)
# They are in a central folder, not with their specific geotif tile sets (at least for now-- we could change this)
adm0_folder = "s3://gfw2-data/gadm_administrative_boundaries/v4.1/v4.1.64__from_gfw-data-lake/raster/epsg-4326/10/40000/adm0/gdal-geotiff/" #GADM v4.1
adm0_zarr_name = "s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/contextual_layer_global_zarr/GADM4_1_adm0_global/20250604/global_GADM41_adm0_20250604.zarr"

pixel_area_folder = "s3://gfw2-data/analyses/umd_area_2013__from_gfw-data-lake/v1.10/raster/epsg-4326/10/40000/area_m/gdal-geotiff/"
pixel_area_zarr_name = "s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/contextual_layer_global_zarr/pixel_area/20250730/global_pixel_area_20250730.zarr"

primary_forest_IFL_folder = "s3://gfw2-data/climate/carbon_model/ifl_primary_merged/processed/20200724/"
primary_forest_IFL_zarr_name = "s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/contextual_layer_global_zarr/IFL2000_tropical_primary_forest_2001/20250806/ifl_primary_forest_merged.zarr"


# Spreadsheet for state_node meanings (local computer and s3 locations)
state_node_lookup_table_local = "/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/LULUCF_state_node_lookup_table.xlsx"
state_node_lookup_table_s3 = "http://gfw2-data.s3.amazonaws.com/climate/AFOLU_flux_model/LULUCF/state_node_lookup_tables/LULUCF_state_node_lookup_table.xlsx"
sheet = "v042_20250805"

pixel_area_interval = "area__ha"

In [ ]:
# %%time

## CREATES ZARRS FOR INPUTS NOT GENERATED BY THE AFOLU MODEL
## THIS SHOULD ONLY EVER HAVE TO BE DONE ONCE FOR EACH INPUT

# print(f"Reading inputs that apply to all intervals: {timestr()}")

# # GADM adm0
# adm0_uris = list_folder_uris(adm0_folder)
# print("adm0_folder:", adm0_folder)
# print(adm0_uris[0])
# print(f"Tile count in {adm0_folder}: {len(adm0_uris)}")

# print(f"   Reading adm0: {timestr()}")
# adm0_xarray_chunks = make_xarray_chunks(adm0_uris, chunk_size)
# adm0_xarray_chunks['band_data'] = adm0_xarray_chunks['band_data'].astype('uint16')  # adm0 should be uint16 but make_xarray_chunks makes it float64 for some reason
# # print("adm0_xarray_chunks:", adm0_xarray_chunks)  # Print to confirm that the zarr datatype is correct

# # print(adm0_xarray_chunks)
# print("X resolution:", np.diff(adm0_xarray_chunks.x.values).mean())
# print("Y resolution:", np.diff(adm0_xarray_chunks.y.values).mean())
# print("X range:", adm0_xarray_chunks.x.values[[0, -1]])
# print("Y range:", adm0_xarray_chunks.y.values[[0, -1]])
# print("Shape (y, x):", adm0_xarray_chunks.sizes['y'], adm0_xarray_chunks.sizes['x'])

# print(f"   zarring adm0: {timestr()}")
# adm0_xarray_chunks.to_zarr(adm0_zarr_name, mode='w')
# print(f"   Finished zarring adm0: {timestr()}")


# # Pixel area
# pixel_area_uris = list_folder_uris(pixel_area_folder)
# print("pixel_area_folder:", pixel_area_folder)
# print(pixel_area_uris[0])
# print(f"Tile count in {pixel_area_folder}: {len(pixel_area_uris)}")

# print(f"   Reading pixel_area: {timestr()}")
# pixel_area_xarray_chunks = make_xarray_chunks(pixel_area_uris, chunk_size)
# print("pixel_area_xarray_chunks:", pixel_area_xarray_chunks)

# print(f"   zarring pixel_area: {timestr()}")
# pixel_area_xarray_chunks.to_zarr(pixel_area_zarr_name, mode='w')
# print(f"   Finished zarring pixel_area: {timestr()}")


# # Humid tropical primary forest/IFL merged
# primary_forest_IFL_uris = list_folder_uris(primary_forest_IFL_folder)
# print("primary_forest_IFL_folder:", primary_forest_IFL_folder)
# print(primary_forest_IFL_uris[0])
# print(f"Tile count in {primary_forest_IFL_folder}: {len(primary_forest_IFL_uris)}")

# print(f"   Reading primary_forest_IFL: {timestr()}")
# primary_forest_IFL_xarray_chunks = make_xarray_chunks(primary_forest_IFL_uris, chunk_size)
# primary_forest_IFL_xarray_chunks['band_data'] = primary_forest_IFL_xarray_chunks['band_data'].astype('uint8')  # should be uint8 but make_xarray_chunks makes it float32 for some reason
# print("primary_forest_IFL_xarray_chunks:", primary_forest_IFL_xarray_chunks)  # Print to confirm that the zarr datatype is correct

# print(f"   zarring primary_forest_IFL: {timestr()}")
# primary_forest_IFL_xarray_chunks.to_zarr(primary_forest_IFL_zarr_name, mode='w')
# print(f"   Finished zarring primary_forest_IFL: {timestr()}")

In [ ]:
# %%time

# ### CONVERTS GEOTIFS TO A SINGLE ZARR (ALL VARIABLES × ALL INTERVALS) AND STORES IT IN S3
# # From https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6899f498-9e9c-8325-aecb-d1d8df7fe48f

# # interval_end_years = [2005]
# # interval_end_years = [2005, 2010, 2015]
# # interval_end_years = [2024]
# # interval_end_years = [2016, 2017, 2018]
# # interval_end_years = [2019, 2020, 2021, 2022, 2023]
# interval_end_years = [2005, 2010, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


# first_interval = True
# analysis_start_time = time.time()

# for interval_end_year in interval_end_years:

#     # Label + numeric bounds
#     if interval_end_year in [2005, 2010, 2015]:
#         interval_label = f"{interval_end_year-4}_{interval_end_year}"
#         interval_start_year = interval_end_year - 4
#     else:
#         interval_label = f"{interval_end_year-1}_{interval_end_year}"
#         interval_start_year = interval_end_year - 1
#     interval = interval_label  # just for printing/logs

#     print(f"Processing {interval}: {timestr()}")
#     interval_start_time = time.time()

#     # URIs for the interval
#     gross_emis_CO2_folder_interval = gross_emis_CO2_folder.replace("INTERVAL", interval)
#     gross_emis_CO2_uris = list_folder_uris(gross_emis_CO2_folder_interval)

#     gross_emis_all_gases_folder_interval = gross_emis_all_gases_folder.replace("INTERVAL", interval)
#     gross_emis_all_gases_uris = list_folder_uris(gross_emis_all_gases_folder_interval)
    
#     gross_remv_all_pools_folder_interval = gross_remv_all_pools_folder.replace("INTERVAL", interval)
#     gross_remv_all_pools_uris = list_folder_uris(gross_remv_all_pools_folder_interval)
    
#     net_flux_all_pools_CO2_folder_interval = net_flux_all_pools_CO2_folder.replace("INTERVAL", interval)
#     net_flux_all_pools_CO2_uris = list_folder_uris(net_flux_all_pools_CO2_folder_interval)

#     net_flux_all_pools_all_gases_folder_interval = net_flux_all_pools_all_gases_folder.replace("INTERVAL", interval)
#     net_flux_all_pools_all_gases_uris = list_folder_uris(net_flux_all_pools_all_gases_folder_interval)
    
#     node_folder_interval = node_folder.replace("INTERVAL", interval)
#     node_tile_year_uris = list_folder_uris(node_folder_interval)

#     print("    gross_emis_CO2_folder_interval:", gross_emis_CO2_folder_interval)
#     print(gross_emis_CO2_uris[0])
#     print(f"    Tile count in {gross_emis_CO2_folder_interval}: {len(gross_emis_CO2_uris)}")

#     print("    gross_emis_all_gases_folder_interval:", gross_emis_all_gases_folder_interval)
#     print(gross_emis_all_gases_uris[0])
#     print(f"    Tile count in {gross_emis_all_gases_folder_interval}: {len(gross_emis_all_gases_uris)}")
    
#     print("    gross_remv_all_pools_folder_interval:", gross_remv_all_pools_folder_interval)
#     print(gross_remv_all_pools_uris[0])
#     print(f"    Tile count in {gross_remv_all_pools_folder_interval}: {len(gross_remv_all_pools_uris)}")

#     print("    net_flux_all_pools_CO2_folder_interval:", net_flux_all_pools_CO2_folder_interval)
#     print(net_flux_all_pools_CO2_uris[0])
#     print(f"    Tile count in {net_flux_all_pools_CO2_folder_interval}: {len(net_flux_all_pools_CO2_uris)}")

#     print("    net_flux_all_pools_all_gases_folder_interval:", net_flux_all_pools_all_gases_folder_interval)
#     print(net_flux_all_pools_all_gases_uris[0])
#     print(f"    Tile count in {net_flux_all_pools_all_gases_folder_interval}: {len(net_flux_all_pools_all_gases_uris)}")

#     print("    node_folder_interval:", node_folder_interval)
#     print(node_tile_year_uris[0])
#     print(f"    Tile count in {node_folder_interval}: {len(node_tile_year_uris)}")

#     # Patterns for analysis layers (no pattern extracted for state_nodes)
#     gross_emis_CO2_output_pattern       = parse_pattern_from_uri(gross_emis_CO2_uris)
#     gross_emis_all_gases_output_pattern = parse_pattern_from_uri(gross_emis_all_gases_uris)
#     gross_remv_all_pools_output_pattern = parse_pattern_from_uri(gross_remv_all_pools_uris)
#     net_flux_CO2_output_pattern         = parse_pattern_from_uri(net_flux_all_pools_CO2_uris)
#     net_flux_all_gases_output_pattern   = parse_pattern_from_uri(net_flux_all_pools_all_gases_uris)

    
#     # Reads geotifs into xarray chunks
#     print(f"   Reading gross emis CO2 only for {interval}: {timestr()}")
#     gross_emis_CO2_xarray_chunks = make_xarray_chunks(gross_emis_CO2_uris, chunk_size)

#     print(f"   Reading gross emis all gases for {interval}: {timestr()}")
#     gross_emis_all_gases_xarray_chunks = make_xarray_chunks(gross_emis_all_gases_uris, chunk_size)

#     print(f"   Reading gross removals for {interval}: {timestr()}")
#     gross_remv_all_pools_xarray_chunks = make_xarray_chunks(gross_remv_all_pools_uris, chunk_size)

#     print(f"   Reading net flux CO2 only for {interval}: {timestr()}")
#     net_flux_all_pools_CO2_xarray_chunks = make_xarray_chunks(net_flux_all_pools_CO2_uris, chunk_size)

#     print(f"   Reading net flux all gases for {interval}: {timestr()}")
#     net_flux_all_pools_all_gases_xarray_chunks = make_xarray_chunks(net_flux_all_pools_all_gases_uris, chunk_size)

#     print(f"   Reading state_nodes for {interval}: {timestr()}")
#     node_xarray_chunks = make_xarray_chunks(node_tile_year_uris, chunk_size)
#     node_xarray_chunks['band_data'] = node_xarray_chunks['band_data'].astype('uint32')  # force uint32

        
#     # Re-chunks to target chunk size
#     target_chunks = {'x': chunk_size, 'y': chunk_size}
#     gross_emis_CO2_xarray_chunks = gross_emis_CO2_xarray_chunks.chunk(target_chunks)
#     gross_emis_all_gases_xarray_chunks = gross_emis_all_gases_xarray_chunks.chunk(target_chunks)
#     gross_remv_all_pools_xarray_chunks = gross_remv_all_pools_xarray_chunks.chunk(target_chunks)
#     net_flux_all_pools_CO2_xarray_chunks = net_flux_all_pools_CO2_xarray_chunks.chunk(target_chunks)
#     net_flux_all_pools_all_gases_xarray_chunks = net_flux_all_pools_all_gases_xarray_chunks.chunk(target_chunks)
#     node_xarray_chunks = node_xarray_chunks.chunk(target_chunks)

    
#     # Builds one Dataset for this interval, adds numeric 'interval' dim, sets chunks, writes/appends to zarr
#     gross_emis_CO2_da           = gross_emis_CO2_xarray_chunks['band_data'].rename(f"{gross_emis_CO2_output_pattern}_yr")
#     gross_emis_all_gases_da     = gross_emis_all_gases_xarray_chunks['band_data'].rename(f"{gross_emis_all_gases_output_pattern}_yr")
#     gross_remv_all_pools_da     = gross_remv_all_pools_xarray_chunks['band_data'].rename(f"{gross_remv_all_pools_output_pattern}_yr")
#     net_flux_all_pools_CO2_da   = net_flux_all_pools_CO2_xarray_chunks['band_data'].rename(f"{net_flux_CO2_output_pattern}_yr")
#     net_flux_all_pools_gases_da = net_flux_all_pools_all_gases_xarray_chunks['band_data'].rename(f"{net_flux_all_gases_output_pattern}_yr")
#     node_da                     = node_xarray_chunks['band_data'].rename('land_state_node').astype('uint32')


#     # Checks that all analysis layers are on the same grid and extent before combining them
#     ref_data = gross_emis_CO2_da
#     for other, name_other in [
#         (gross_emis_all_gases_da,     gross_emis_all_gases_output_pattern),
#         (gross_remv_all_pools_da,     gross_remv_all_pools_output_pattern),
#         (net_flux_all_pools_CO2_da,   net_flux_CO2_output_pattern),
#         (net_flux_all_pools_gases_da, net_flux_all_gases_output_pattern),
#         (node_da,                     node_output_pattern),
#     ]:
#         assert_same_grid(ref_data, other, gross_emis_CO2_output_pattern, name_other)

#     # Merges model outputs into a single Dataset for this interval
#     ds_interval = xr.merge([
#         gross_emis_CO2_da.to_dataset(),
#         gross_emis_all_gases_da.to_dataset(),
#         gross_remv_all_pools_da.to_dataset(),
#         net_flux_all_pools_CO2_da.to_dataset(),
#         net_flux_all_pools_gases_da.to_dataset(),
#         node_da.to_dataset(),
#     ])

#     # Adds numeric interval dimension + numeric coords (strings not allowed).
#     # Using the interval END year as the coordinate index keeps it simple and stable.
#     ds_interval = ds_interval.expand_dims(interval=[interval_end_year])
#     ds_interval = ds_interval.assign_coords(
#         interval_start_year=("interval", [interval_start_year]),
#         interval_end_year=("interval", [interval_end_year]),
#     )

#     # Set per-variable encodings to control zarr chunking (interval=1, y/x=chunk_size)
#     for v in ds_interval.data_vars:
#         dims = ds_interval[v].dims  # typically ('interval','y','x')
#         chunks = []
#         for d in dims:
#             if d == 'interval':
#                 chunks.append(1)
#             elif d in ('y', 'x'):
#                 chunks.append(chunk_size)  # e.g., 10000
#             else:
#                 chunks.append(1)
#         ds_interval[v].encoding = {'chunks': tuple(chunks)}

#     # Writes or appends to the single global Zarr
#     if first_interval:
#         print(f"   Creating all-intervals zarr: {all_intervals_zarr}")
#         write_or_append_with_retry(ds_interval, all_intervals_zarr, append=False, max_retries=3, wait_seconds=5)
#         print("   Created global Zarr with first interval")
#         first_interval = False
#     else:
#         print(f"   Appending interval {interval_label}: {all_intervals_zarr}")
#         write_or_append_with_retry(ds_interval, all_intervals_zarr, append=True, max_retries=3, wait_seconds=5)
#         print(f"   Appended interval {interval_label}")

#     interval_end_time = time.time()
#     print(f"   {interval_label} took {round(interval_end_time - interval_start_time)} seconds")

# print(f"Done zarring all intervals into: {all_intervals_zarr}  {timestr()}")
# analysis_end_time = time.time()
# print(f"Analysis took {round(analysis_end_time - analysis_start_time)} seconds")

# try:
#     client.shutdown()
# except Exception:
#     pass

In [130]:
%%time

### RUNS ZONAL STATISTICS ANALYSIS (reads single global Zarr across all intervals)
# From https://chatgpt.com/g/g-vK4oPfjfp-coding-assistant/c/6899f498-9e9c-8325-aecb-d1d8df7fe48f

combined_df = pd.DataFrame()  # dataframe for outputs across all model intervals
analysis_start_time = time.time()

# interval_end_years = [2005]
# interval_end_years = [2005, 2010, 2015]
# interval_end_years = [2024]
# interval_end_years = [2016, 2017, 2018]
# interval_end_years = [2019, 2020, 2021, 2022, 2023]
interval_end_years = [2005, 2010, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

# Creates dataframe of state_node codes and meanings
state_node_df = create_state_node_df(state_node_lookup_table_local, state_node_lookup_table_s3, sheet)
node_codes = np.array(list(state_node_df['state_nodes']), dtype=np.uint32)

# ADD ALL CONTEXTUAL LAYER NAMES HERE. THEY ARE USED IN DATAFRAME CREATION.
contextual_layer_names = ['state_nodes', 'gadm_adm0', 'primary_forest_IFL']

# Opens all inputs not from the model here
print("Opening zarrs for non-model inputs")
pixel_area = xr.open_zarr(pixel_area_zarr_name).band_data
adm0 = xr.open_zarr(adm0_zarr_name).band_data
primary_forest_IFL = xr.open_zarr(primary_forest_IFL_zarr_name).band_data

# Opens the single global zarr of all model outputs (all intervals × all variables)
all_intervals_zarr = f"{zarr_s3_path}all_intervals/afolu_flux__all_layers__all_intervals.zarr"
print(f"Opening full model output zarr: {all_intervals_zarr}")
ds_all_global = xr.open_zarr(all_intervals_zarr, storage_options={"anon": False})

# Zonal stats iterates through selected intervals
for interval_end_year in interval_end_years:

    # Human-readable label for logs only
    if interval_end_year in [2005, 2010, 2015]:
        interval_label = f"{interval_end_year-4}_{interval_end_year}"
        interval_length = 5
    else:
        interval_label = f"{interval_end_year-1}_{interval_end_year}"
        interval_length = 1

    print(f"Processing {interval_label}: {timestr()}")
    interval_start_time = time.time()

    # (Optional) still compute filename patterns from the GeoTIFF URIs if you rely on them later
    gross_emis_CO2_folder_interval = gross_emis_CO2_folder.replace("INTERVAL", interval_label)
    gross_emis_CO2_uris = list_folder_uris(gross_emis_CO2_folder_interval)
    gross_emis_all_gases_folder_interval = gross_emis_all_gases_folder.replace("INTERVAL", interval_label)
    gross_emis_all_gases_uris = list_folder_uris(gross_emis_all_gases_folder_interval)
    gross_remv_all_pools_folder_interval = gross_remv_all_pools_folder.replace("INTERVAL", interval_label)
    gross_remv_all_pools_uris = list_folder_uris(gross_remv_all_pools_folder_interval)
    net_flux_all_pools_CO2_folder_interval = net_flux_all_pools_CO2_folder.replace("INTERVAL", interval_label)
    net_flux_all_pools_CO2_uris = list_folder_uris(net_flux_all_pools_CO2_folder_interval)
    net_flux_all_pools_all_gases_folder_interval = net_flux_all_pools_all_gases_folder.replace("INTERVAL", interval_label)
    net_flux_all_pools_all_gases_uris = list_folder_uris(net_flux_all_pools_all_gases_folder_interval)
    node_folder_interval = node_folder.replace("INTERVAL", interval_label)
    node_tile_year_uris = list_folder_uris(node_folder_interval)

    gross_emis_CO2_output_pattern       = f"{parse_pattern_from_uri(gross_emis_CO2_uris)}_yr"
    gross_emis_all_gases_output_pattern = f"{parse_pattern_from_uri(gross_emis_all_gases_uris)}_yr"
    gross_remv_all_pools_output_pattern = f"{parse_pattern_from_uri(gross_remv_all_pools_uris)}_yr"
    net_flux_CO2_output_pattern         = f"{parse_pattern_from_uri(net_flux_all_pools_CO2_uris)}_yr"
    net_flux_all_gases_output_pattern   = f"{parse_pattern_from_uri(net_flux_all_pools_all_gases_uris)}_yr"
    node_output_pattern                 = parse_pattern_from_uri(node_tile_year_uris)

    # Selecta this interval from the global zarr (as designated by numeric end-year of interval, since zarr interval has to be numeric)
    print(f"   Selecting interval={interval_end_year} from global Zarr: {timestr()}")
    ds_interval = ds_all_global.sel(interval=interval_end_year)

    # Pulls variables out by name (now 2D y/x after the selection)
    gross_emis_CO2               = ds_interval[gross_emis_CO2_output_pattern]
    gross_emis_all_gases         = ds_interval[gross_emis_all_gases_output_pattern]
    gross_remv_all_pools         = ds_interval[gross_remv_all_pools_output_pattern]
    net_flux_all_pools_CO2       = ds_interval[net_flux_CO2_output_pattern]
    net_flux_all_pools_all_gases = ds_interval[net_flux_all_gases_output_pattern]

    # State nodes as the reference grid
    state_nodes = ds_interval["land_state_node"].astype("uint32")
    state_nodes.name = "state_nodes"

    # Align/crop everything to state_nodes
    print(f"   Aligning {interval_label}: {timestr()}")
    reference = state_nodes
    state_nodes_aligned         = reference
    adm0_aligned                = safe_crop(adm0, reference)
    primary_forest_IFL_aligned  = safe_crop(primary_forest_IFL, reference)
    pixel_area_aligned          = safe_crop(pixel_area, reference)

    gross_emis_CO2_aligned               = safe_crop(gross_emis_CO2, reference)
    gross_emis_all_gases_aligned         = safe_crop(gross_emis_all_gases, reference)
    gross_remv_all_pools_aligned         = safe_crop(gross_remv_all_pools, reference)
    net_flux_all_pools_CO2_aligned       = safe_crop(net_flux_all_pools_CO2, reference)
    net_flux_all_pools_all_gases_aligned = safe_crop(net_flux_all_pools_all_gases, reference)

    # Pixel area in hectares
    pixel_area__ha = (pixel_area_aligned / 10000).astype("float32").rename(pixel_area_interval)

    # Build flux cube (stacked layers)
    flux_cube = xr.DataArray(
        dask.array.stack([
            (gross_emis_CO2_aligned.data               * pixel_area__ha.data).astype("float32"),
            (gross_emis_all_gases_aligned.data         * pixel_area__ha.data).astype("float32"),
            (gross_remv_all_pools_aligned.data         * pixel_area__ha.data).astype("float32"),
            (net_flux_all_pools_CO2_aligned.data       * pixel_area__ha.data).astype("float32"),
            (net_flux_all_pools_all_gases_aligned.data * pixel_area__ha.data).astype("float32"),
            pixel_area__ha.data,
        ]),
        dims=("analysis_layer", "y", "x"),
    )

    # Final alignment (helps when zarr comes from non-contiguous tiles)
    flux_cube, adm0_aligned, state_nodes_aligned, primary_forest_IFL_aligned = xr.align(
        flux_cube, adm0_aligned, state_nodes_aligned, primary_forest_IFL_aligned, join="override"
    )

    # Unique names for contextual layers
    adm0_aligned.name = "gadm_adm0"
    primary_forest_IFL_aligned.name = "primary_forest_IFL"
    state_nodes_aligned.name = "state_nodes"

    # # Also name analysis layers (optional/for metadata)
    # pixel_area__ha.name = pixel_area_interval
    # gross_emis_CO2.name = gross_emis_CO2_output_pattern
    # gross_emis_all_gases.name = gross_emis_all_gases_output_pattern
    # gross_remv_all_pools.name = gross_remv_all_pools_output_pattern
    # net_flux_all_pools_CO2.name = net_flux_CO2_output_pattern  
    # net_flux_all_pools_all_gases.name = net_flux_all_gases_output_pattern

    # # (Optional) print metadata
    # input_layers = [
    #     adm0_aligned, primary_forest_IFL_aligned, state_nodes_aligned,
    #     pixel_area__ha, gross_emis_CO2, gross_emis_all_gases,
    #     gross_remv_all_pools, net_flux_all_pools_CO2, net_flux_all_pools_all_gases
    # ]
    # for ds in input_layers:
    #     print(f"   Metadata for {ds.name} for {interval_label}")
    #     print(f"     X, Y resolution: {np.diff(ds.x.values).mean(), np.diff(ds.y.values).mean()}")
    #     print(f"     X range: {ds.x.values[[0, -1]]}")
    #     print(f"     Y range: {ds.y.values[[0, -1]]}")
    #     print(f"     Shape (x, y): {ds.sizes['x'], ds.sizes['y']}")
    #     print(f"     Data type: {ds.dtype}")

    # Compute zonal stats
    print(f"   Computing {interval_label}: {timestr()}")
    flux_results = xarray_reduce(
        flux_cube,  # Layers to be analyzed
        *(adm0_aligned, state_nodes_aligned, primary_forest_IFL_aligned),  # Contextual layers
        func='sum',
        expected_groups=(gadm_adm0_ids, node_codes, primary_forest_IFL_codes),
        reindex=ReindexStrategy(blockwise=False, array_type=ReindexArrayType.SPARSE_COO),
        fill_value=0
    ).compute()

    
    # Analysis-layer name mapping for the output dataframe 
    analysis_layer_dict = {
        0: gross_emis_CO2_output_pattern,
        1: gross_emis_all_gases_output_pattern,
        2: gross_remv_all_pools_output_pattern, 
        3: net_flux_CO2_output_pattern,
        4: net_flux_all_gases_output_pattern,
        5: pixel_area_interval,
    }

    # Prepare outputs → dataframe
    coord_dict = convert_to_coord_dict(flux_results, interval_label)
    df = create_interval_df(coord_dict, state_node_df, analysis_layer_dict, interval_length, interval_end_year)
    df = calculate_interval_flux_densities(df)

    # Combine with previous intervals
    combined_df = pd.concat([combined_df, df])

    interval_end_time = time.time()
    print(f"   {interval_label} took {round(interval_end_time - interval_start_time)} seconds")

combined_df = combined_df.reset_index(drop=True)

analysis_end_time = time.time()
print(f"Analysis took {round(analysis_end_time - analysis_start_time)} seconds")

# print(combined_df)

# try:
#     client.shutdown()
# except Exception:
#     pass

Failed to download file from S3. Falling back to local file. Error: HTTPConnectionPool(host='gfw2-data.s3.amazonaws.com', port=80): Max retries exceeded with url: /climate/AFOLU_flux_model/LULUCF/state_node_lookup_tables/LULUCF_state_node_lookup_table.xlsx (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7f8674558830>: Failed to establish a new connection: [Errno 111] Connection refused'))
Reading file from local path: /mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/LULUCF_state_node_lookup_table.xlsx
Opening zarrs for non-model inputs
Opening full model output zarr: s3://gfw2-data/climate/AFOLU_flux_model/LULUCF/outputs/version_0_4_2_1x1_inputs_per_ha_all_yrs_combined_zarr/zarr/20250806/all_intervals/afolu_flux__all_layers__all_intervals.zarr
Processing 2001_2005: 20250814_10_30_18
   Selecting interval=2005 from global Zarr: 20250814_10_30_18
   Aligning 2001_2005: 20250814_10_30_18
   Computing 2001_2005: 20250814_10_30_19
   Postprocessing 2001_2005: 202

In [134]:
### Flux check
# Gross emissions all pools all gases 2005 should = 458509003
# Area 2005 should = 206902007
# Net flux all pools all gases 2010 should = 94761495
# Gross removals 2024 should = -586198699
print(combined_df[(combined_df.analysis_layer == gross_emis_all_gases_output_pattern) & (combined_df.interval_end == 2005)]['value'].sum().round())
print(combined_df[(combined_df.analysis_layer == pixel_area_interval) & (combined_df.interval_end == 2005)]['value'].sum().round())
print(combined_df[(combined_df.analysis_layer == net_flux_all_gases_output_pattern) & (combined_df.interval_end == 2010)]['value'].sum().round())
print(combined_df[(combined_df.analysis_layer == gross_remv_all_pools_output_pattern) & (combined_df.interval_end == 2024)]['value'].sum().round())

458509060.0
206902020.0
94761510.0
-586198700.0


In [132]:
### Emission factor check
# EF for 31120000 for 2005 should = 15.87
# EF for 32139000 for 2005 should = 18.99
print(combined_df[(combined_df.analysis_layer == gross_emis_all_gases_output_pattern) & (combined_df.interval_end == 2005) & (combined_df.state_nodes == 31120000)]['value'].sum() / 
      (combined_df[(combined_df.analysis_layer == pixel_area_interval) & (combined_df.interval_end == 2005) & (combined_df.state_nodes == 31120000)]['value'].sum()) / C_to_CO2)
print(combined_df[(combined_df.analysis_layer == gross_emis_all_gases_output_pattern) & (combined_df.interval_end == 2005) & (combined_df.state_nodes == 32139000)]['value'].sum() / 
      (combined_df[(combined_df.analysis_layer == pixel_area_interval) & (combined_df.interval_end == 2005) & (combined_df.state_nodes == 32139000)]['value'].sum()) / C_to_CO2)

15.874313
18.998857


In [ ]:
# coord_dict = convert_to_coord_dict(flux_results, interval_label)
# df = create_interval_df(coord_dict, state_node_df, analysis_layer_dict, interval_length, interval_end_year)
# combined_df = calculate_interval_flux_densities(df)
# combined_df

In [133]:
combined_df_wide = combined_df.pivot(index=['state_nodes', 'broad_class', 'detailed_class', 'interval_end', 'gadm_adm0', 'primary_forest_IFL', 'meaning'], 
                                     columns="analysis_layer", values="value").reset_index()
combined_df_wide

analysis_layer,state_nodes,broad_class,detailed_class,interval_end,gadm_adm0,primary_forest_IFL,meaning,area__ha,gross_emissions__all_C_pools__CO2_only__MgCO2_yr,gross_emissions__all_C_pools__CO2_only__MgCO2_yr__C_per_ha,gross_emissions__all_C_pools__all_gases__MgCO2e_yr,gross_emissions__all_C_pools__all_gases__MgCO2e_yr__C_per_ha,gross_removals__all_C_pools__MgCO2_yr,gross_removals__all_C_pools__MgCO2_yr__C_per_ha,net_flux__all_C_pools__CO2_only__MgCO2_yr,net_flux__all_C_pools__CO2_only__MgCO2_yr__C_per_ha,net_flux__all_C_pools__all_gases__MgCO2e_yr,net_flux__all_C_pools__all_gases__MgCO2e_yr__C_per_ha
0,21100000,tree,tree_gain,2001,BRA,0,Gain of oil palm (incl. SDPT oil palm),1.315250e+02,NaN,NaN,NaN,NaN,-875.009521,-1.8144,-875.009521,-1.8144,-875.009521,-1.8144
1,21100000,tree,tree_gain,2002,BRA,0,Gain of oil palm (incl. SDPT oil palm),1.315250e+02,NaN,NaN,NaN,NaN,-875.009521,-1.8144,-875.009521,-1.8144,-875.009521,-1.8144
2,21100000,tree,tree_gain,2003,BRA,0,Gain of oil palm (incl. SDPT oil palm),1.315250e+02,NaN,NaN,NaN,NaN,-875.009521,-1.8144,-875.009521,-1.8144,-875.009521,-1.8144
3,21100000,tree,tree_gain,2004,BRA,0,Gain of oil palm (incl. SDPT oil palm),1.315250e+02,NaN,NaN,NaN,NaN,-875.009521,-1.8144,-875.009521,-1.8144,-875.009521,-1.8144
4,21100000,tree,tree_gain,2005,BRA,0,Gain of oil palm (incl. SDPT oil palm),1.315250e+02,NaN,NaN,NaN,NaN,-875.009521,-1.8144,-875.009521,-1.8144,-875.009521,-1.8144
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5509,70000000,no_flux,no_flux,2023,PRY,1,Not in decision tree,5.626628e+01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5510,70000000,no_flux,no_flux,2024,BRA,0,Not in decision tree,8.013739e+06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5511,70000000,no_flux,no_flux,2024,BRA,1,Not in decision tree,5.211972e+04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5512,70000000,no_flux,no_flux,2024,PRY,0,Not in decision tree,1.753770e+04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [137]:
# Export to a csv so data can be used in Excel or reused
combined_df_wide.to_csv('Cerrado_model_v0_4_2__2000_2024__20250814.csv', index=False)

In [ ]:
walker = pyg.walk(combined_df_wide)

In [ ]:
vis_spec = r"""{"config":[{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1},"encodings":{"dimensions":[{"dragId":"gw_6iQt","fid":"broad_class","name":"broad_class","basename":"broad_class","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_A3G-","fid":"detailed_class","name":"detailed_class","basename":"detailed_class","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_Ni12","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"quantitative","analyticType":"dimension"},{"dragId":"gw_5Njj","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_tQV9","fid":"primary_forest_IFL","name":"primary_forest_IFL","basename":"primary_forest_IFL","semanticType":"ordinal","analyticType":"dimension"},{"dragId":"gw_K7GB","fid":"state_nodes","name":"state_nodes","basename":"state_nodes","analyticType":"dimension","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_6_au","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_mea_key_fid","fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"}],"measures":[{"dragId":"gw_G-rP","fid":"area__ha","name":"area__ha","basename":"area__ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_RlYr","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2_yr","name":"gross_emissions__all_C_pools__CO2_only__MgCO2_yr","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2_yr","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_jOut","fid":"gross_emissions__all_C_pools__CO2_only__MgCO2_yr__C_per_ha","name":"gross_emissions__all_C_pools__CO2_only__MgCO2_yr__C_per_ha","basename":"gross_emissions__all_C_pools__CO2_only__MgCO2_yr__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_6BBB","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e_yr","name":"gross_emissions__all_C_pools__all_gases__MgCO2e_yr","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e_yr","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_-Wbd","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e_yr__C_per_ha","name":"gross_emissions__all_C_pools__all_gases__MgCO2e_yr__C_per_ha","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e_yr__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_dksR","fid":"gross_removals__all_C_pools__MgCO2_yr","name":"gross_removals__all_C_pools__MgCO2_yr","basename":"gross_removals__all_C_pools__MgCO2_yr","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_Krnp","fid":"gross_removals__all_C_pools__MgCO2_yr__C_per_ha","name":"gross_removals__all_C_pools__MgCO2_yr__C_per_ha","basename":"gross_removals__all_C_pools__MgCO2_yr__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_B1BS","fid":"net_flux__all_C_pools__CO2_only__MgCO2_yr","name":"net_flux__all_C_pools__CO2_only__MgCO2_yr","basename":"net_flux__all_C_pools__CO2_only__MgCO2_yr","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_FhHS","fid":"net_flux__all_C_pools__CO2_only__MgCO2_yr__C_per_ha","name":"net_flux__all_C_pools__CO2_only__MgCO2_yr__C_per_ha","basename":"net_flux__all_C_pools__CO2_only__MgCO2_yr__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_VTKb","fid":"net_flux__all_C_pools__all_gases__MgCO2e_yr","name":"net_flux__all_C_pools__all_gases__MgCO2e_yr","basename":"net_flux__all_C_pools__all_gases__MgCO2e_yr","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_RRzu","fid":"net_flux__all_C_pools__all_gases__MgCO2e_yr__C_per_ha","name":"net_flux__all_C_pools__all_gases__MgCO2e_yr__C_per_ha","basename":"net_flux__all_C_pools__all_gases__MgCO2e_yr__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_count_fid","fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"dragId":"gw_mea_val_fid","fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"rows":[{"dragId":"gw_rSli","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e_yr","name":"gross_emissions__all_C_pools__all_gases__MgCO2e_yr","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e_yr","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_JMDG","fid":"gross_emissions__all_C_pools__all_gases__MgCO2e_yr__C_per_ha","name":"gross_emissions__all_C_pools__all_gases__MgCO2e_yr__C_per_ha","basename":"gross_emissions__all_C_pools__all_gases__MgCO2e_yr__C_per_ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"},{"dragId":"gw_Ejh-","fid":"area__ha","name":"area__ha","basename":"area__ha","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"columns":[{"dragId":"gw_Iwcp","fid":"meaning","name":"meaning","basename":"meaning","semanticType":"nominal","analyticType":"dimension"},{"dragId":"gw_5b5G","fid":"interval_end","name":"interval_end","basename":"interval_end","semanticType":"quantitative","analyticType":"dimension"}],"color":[],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[{"dragId":"gw_VkFs","fid":"primary_forest_IFL","name":"primary_forest_IFL","basename":"primary_forest_IFL","semanticType":"ordinal","analyticType":"dimension","rule":{"type":"not in","value":[1]}},{"dragId":"gw_3sWa","fid":"gadm_adm0","name":"gadm_adm0","basename":"gadm_adm0","semanticType":"nominal","analyticType":"dimension","rule":{"type":"not in","value":["PRY"]}}],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_7ODb","name":"Chart 1"}],"chart_map":{},"workflow_list":[{"workflow":[{"type":"filter","filters":[{"fid":"primary_forest_IFL","rule":{"type":"not in","value":[1]}},{"fid":"gadm_adm0","rule":{"type":"not in","value":["PRY"]}}]},{"type":"view","query":[{"op":"aggregate","groupBy":["meaning","interval_end"],"measures":[{"field":"gross_emissions__all_C_pools__all_gases__MgCO2e_yr","agg":"sum","asFieldKey":"gross_emissions__all_C_pools__all_gases__MgCO2e_yr_sum"},{"field":"gross_emissions__all_C_pools__all_gases__MgCO2e_yr__C_per_ha","agg":"sum","asFieldKey":"gross_emissions__all_C_pools__all_gases__MgCO2e_yr__C_per_ha_sum"},{"field":"area__ha","agg":"sum","asFieldKey":"area__ha_sum"}]}]}]}],"timezoneOffsetSeconds":-14400,"version":"0.3.17"}"""
pyg.walk(combined_df_wide, spec=vis_spec)

In [135]:
combined_df_wide_renamed = combined_df_wide.rename(columns={"gross_emissions__all_C_pools__all_gases__MgCO2e_yr":"emis__all_C__all_gases__MgCO2e_yr", 
                                         "gross_emissions__all_C_pools__all_gases__MgCO2e_yr__C_per_ha":"emis__all_C__all_gases__MgCO2e_yr_ha"})

In [136]:
walker = pyg.walk(combined_df_wide_renamed)

Box(children=(HTML(value='<div id="ifr-pyg-2" style="height: auto">\n    <head>\n        <meta http-equiv="Con…